# Fine-tune a compact embedding model for language- and domain-specific product retrieval on Amazon SageMaker

Semantic product search uses text embeddings to match what a shopper types to the right
items in a catalog. General-purpose embedding models, such as Amazon Titan Text
Embeddings on Amazon Bedrock, provide multilingual representations and are a managed
starting point that requires no training. At the same time, embedding quality is task-
and language-specific.

In this notebook, we specialize a compact, open embedding model for one task (product
retrieval) in one language (Brazilian-Portuguese, pt-BR) by fine-tuning it on domain
data with Amazon Translate, Amazon Bedrock, and Amazon SageMaker. The approach is
language-agnostic and reproduces for any language by changing the Amazon Translate
target. We use Amazon Titan Text Embeddings V2 as a general-purpose reference point to
quantify how much task- and language-specific fine-tuning adds here.

## Situation

General-purpose models are trained for broad coverage across many tasks and languages,
which is useful when you need a model with no training effort. For a narrow task such as
product retrieval in a specific language, the domain vocabulary, brand names, and short
or abbreviated e-commerce queries may benefit from representations tuned to that task and
language. Vector dimensionality also matters operationally, since larger vectors increase
index size, memory, and serving cost. A large human-annotated product-search benchmark
already exists, the **Shopping Queries Dataset (ESCI)** from Amazon Science
(`amazon-research/esci-code`, Apache-2.0, arXiv:2206.06588); its text is in English,
Spanish, and Japanese, so we translate a subset to pt-BR.

## Task

Answer a practical question for a pt-BR retail use case: **how much can task- and
language-specific fine-tuning improve a compact, open embedding model on product
retrieval?**

- **Model we fine-tune:** `intfloat/multilingual-e5-small` (118M params, 384-dim, MIT).
- **General-purpose reference point:** Amazon Titan Text Embeddings V2 on Amazon Bedrock
  (1,024-dim, multilingual), used to contextualize the gains.
- **Metric:** full-catalog retrieval on a held-out test set (nDCG@k, Recall@k).

Embedding suitability depends on task and language, so this is not about one model being
better in general; it shows how much specializing a model for a specific task and
language changes results on that task.

## Action (solution overview)

1. Get the ESCI dataset from Amazon Science
2. Sample a controlled 10,000-query subset
3. Translate EN -> pt-BR with Amazon Translate (+ brand custom terminology)
4. Build a labelled dataset and a query-level train/test/validation split
5. Build the evaluation harness (graded-relevance metrics, full-catalog retrieval)
6. Establish reference points: base e5-small and Titan (full corpus)
7. Fine-tune e5-small on Amazon SageMaker
8. Compare

**Expected result (full-corpus retrieval, test set):**

| Model | dims | nDCG@10 | Recall@10 |
| --- | --- | --- | --- |
| e5-small (base) | 384 | 0.276 | 0.345 |
| **e5-small (fine-tuned)** | **384** | **0.432** | **0.528** |
| Titan Text Embeddings V2 | 1024 | 0.373 | 0.458 |

> **Costs & prerequisites.** Steps 3, 6 and 7 use AWS (Amazon Translate, Amazon Bedrock,
> Amazon SageMaker) and incur charges. Translating the full sample is ~US$947; the
> SageMaker fine-tuning job is ~US$0.27; the Titan embeddings are cents. Set `N_QUERIES`
> small in step 2 for a low-cost trial, and scope the IAM roles you create to the S3
> prefixes and API actions this notebook actually calls. Prices vary by AWS Region and
> change over time; check the AWS pricing pages before running the paid track.

In [ ]:
# =============================================================================
# DATA_MODE  ->  pick ONE of the two tracks
# =============================================================================
#   "bundled" (default, FREE, no AWS needed for data prep)
#       Loads the pre-translated pt-BR sample shipped in this repository
#       (data/sample_dataset_ptbr.sample.jsonl -- 500 queries / 3,072
#       products). Steps 1-4 are SKIPPED. Use this to run the notebook
#       end to end and see the workflow work.
#
#   "full"  (PAID, regenerates the pt-BR data from the original ESCI)
#       Runs steps 1-4: clone the ~1 GB English ESCI dataset, sample
#       N_QUERIES, translate EN -> pt-BR with Amazon Translate, and
#       reassemble. Translating the full 10,000-query sample costs about
#       US$947. Every cell in steps 1-4 carries a banner saying it belongs
#       to this track and is skipped otherwise.
# =============================================================================
DATA_MODE = "bundled"          # "bundled" | "full"
BUNDLED_SAMPLE = "data/sample_dataset_ptbr.sample.jsonl"

assert DATA_MODE in ("bundled", "full"), 'DATA_MODE must be "bundled" or "full"'

# Optional steps. Both default to True; set either to False to skip that step.
#   RUN_TITAN     -> step 6, the Amazon Titan Text Embeddings V2 reference point.
#                    Needs Amazon Bedrock model access; the comparison table just
#                    omits that row when skipped.
#   RUN_TRAINING  -> step 7.2, the Amazon SageMaker fine-tuning job (~US$0.27).
#                    Needs S3_BUCKET and a SageMaker execution role.
RUN_TITAN = True
RUN_TRAINING = True

# Dependencies (uncomment to install)
# !pip install -q sentence-transformers pandas pyarrow numpy boto3 xlsxwriter openpyxl sagemaker

import os, json, random, time
import boto3          # used by the Titan embedder (step 6) in both DATA_MODEs
import numpy as np

# Reproducibility: a fixed seed makes the sampling, the train/test/validation
# split, and hard-negative selection deterministic, so you get the same subset
# and near-identical metrics on re-runs. Model fine-tuning also seeds torch (see
# the training script), so fine-tuned numbers are reproducible up to minor GPU
# nondeterminism.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =============================================================================
# EDIT THESE  ->  the cell raises an error until you replace every REPLACE_ME.
# =============================================================================
REGION = "us-east-1"
S3_BUCKET = "REPLACE_ME"            # an S3 bucket you own, e.g. "my-embeddings-bucket"
TRANSLATE_ROLE_ARN = "REPLACE_ME"  # IAM role ARN that Amazon Translate can assume for S3
SAGEMAKER_ROLE_ARN = "REPLACE_ME"  # SageMaker execution role ARN (auto-detected in Studio)

# ---- COST ALERT (read before running Step 3) --------------------------------
# Amazon Translate bills per source character ($15 / million characters).
# The published results use N_QUERIES = 10,000, and translating that full sample costs
# approximately US$947 (about 63.2M characters). The cost scales with N_QUERIES.
# To try the workflow cheaply, set N_QUERIES LOW (for example, 500) in Step 2
# before running the translation step.
# -----------------------------------------------------------------------------

# In "bundled" mode only SAGEMAKER_ROLE_ARN + S3_BUCKET matter (step 7),
# and only if you actually run the fine-tuning job outside SageMaker Studio.
_needed = (("S3_BUCKET", "TRANSLATE_ROLE_ARN", "SAGEMAKER_ROLE_ARN")
           if DATA_MODE == "full" else ())
_missing = [n for n in _needed if globals()[n] == "REPLACE_ME"]
if _missing:
    raise ValueError(
        "Replace these placeholder config values before continuing: "
        + ", ".join(_missing)
        + ". Also review the COST ALERT above before starting the translation step."
    )
print(f"Config OK  |  mode={DATA_MODE}  region={REGION}  seed={SEED}")

## 1. Get the dataset

The **Shopping Queries Dataset (ESCI)** is released publicly by Amazon Science under
Apache-2.0 (repo: `amazon-research/esci-code`, paper: arXiv:2206.06588). It contains
customer queries paired with products, each judged **E**xact / **S**ubstitute /
**C**omplement / **I**rrelevant, in English, Spanish and Japanese.

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 1: download the original English ESCI dataset
# Clones ~1 GB of parquet from amazon-research/esci-code. Free, but only
# needed if you are regenerating the pt-BR data from the English source.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 1: download the original English ESCI dataset (using bundled pt-BR sample).')
else:
    # Clone the repo to get the parquet files (~1 GB products file).
    #
    # These .parquet files are tracked with Git LFS. If the git-lfs client is
    # not installed, `git clone` still exits 0 but leaves ~130-byte POINTER
    # STUBS on disk instead of the real data, and the failure would only show
    # up later as an unreadable-parquet error. So we check the file sizes.
    import shutil, subprocess

    LFS_MIN_BYTES = 1_000_000   # a real parquet is >= 50 MB; a pointer is ~130 B

    if shutil.which("git") is None:
        raise RuntimeError("git not found on PATH -- required to fetch the ESCI dataset.")

    _has_lfs = subprocess.run(["git", "lfs", "version"],
                              capture_output=True).returncode == 0
    if not _has_lfs:
        print("WARNING: git-lfs is NOT installed. The ESCI .parquet files are "
              "Git LFS objects and will download as small pointer stubs.\n"
              "         Install it first: 'brew install git-lfs' (macOS), "
              "'apt-get install git-lfs' (Debian/Ubuntu),\n"
              "         or 'conda install -c conda-forge git-lfs', then run "
              "'git lfs install' and delete the esci-code/ folder.")

    if not os.path.exists("esci-code"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/amazon-research/esci-code.git"],
                       check=True)

    DATA = "esci-code/shopping_queries_dataset"
    EXAMPLES = f"{DATA}/shopping_queries_dataset_examples.parquet"
    PRODUCTS = f"{DATA}/shopping_queries_dataset_products.parquet"

    # Existence is not enough: a pointer stub also "exists". Check the size.
    _stubs = [p for p in (EXAMPLES, PRODUCTS)
              if not os.path.exists(p) or os.path.getsize(p) < LFS_MIN_BYTES]
    if _stubs:
        _detail = "\n".join(
            f"    {p}: "
            + (f"{os.path.getsize(p):,} bytes (Git LFS pointer, not the real file)"
               if os.path.exists(p) else "missing")
            for p in _stubs)
        raise RuntimeError(
            "The ESCI parquet files did not download properly:\n" + _detail
            + "\n\nThis is almost always a missing git-lfs client. Fix it with:\n"
              "    1) install git-lfs (see the warning above)\n"
              "    2) git lfs install\n"
              "    3) rm -rf esci-code\n"
              "    4) re-run this cell\n"
              "Alternatively, set DATA_MODE = \"bundled\" to use the "
              "pre-translated sample and skip this step entirely.")

    print(f"examples: {os.path.getsize(EXAMPLES):,} bytes | "
          f"products: {os.path.getsize(PRODUCTS):,} bytes  (git-lfs OK)")


In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 1: load the ESCI parquet files
# Reads the English parquet files into pandas. Needs step 1's clone.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 1: load the ESCI parquet files (using bundled pt-BR sample).')
else:
    import pandas as pd
    df_examples = pd.read_parquet(EXAMPLES)
    df_products = pd.read_parquet(PRODUCTS)
    print(f"examples: {len(df_examples):,} rows")
    print(f"products: {len(df_products):,} rows")
    df_examples.head(3)

## 2. Sample a controlled 10k-query subset

We restrict to the **US/English** locale and the **E/S/I** labels, then select 10,000
queries with a controlled per-query label mix. Because the data is dominated by Exact
matches, we use a **trim-based** strategy: a query qualifies if it can reach the
minimums (E>=1, I>=2), and we then randomly keep a target count per label within a
range (E 1-4, S 0-2, I 2-7). Everything is seeded for reproducibility.

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 2: sample a controlled subset of English queries
# Selects N_QUERIES from the ENGLISH data. The bundled sample already
# contains an equivalent 500-query slice, already translated.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 2: sample a controlled subset of English queries (using bundled pt-BR sample).')
else:
    N_QUERIES = 10_000                 # reduce (e.g. 500) for a cheap trial run
    LABEL_RULES = {"E": (1, 4), "S": (0, 2), "I": (2, 7)}
    KEEP = list(LABEL_RULES)

    m = (df_examples["product_locale"] == "us") & (df_examples["esci_label"].isin(KEEP))
    df_esi = df_examples.loc[m].copy()

    counts = df_esi.groupby(["query_id", "esci_label"]).size().unstack(fill_value=0)
    for l in KEEP:
        if l not in counts.columns:
            counts[l] = 0
    eligible = counts.index[(counts["E"] >= 1) & (counts["I"] >= 2)].to_numpy()
    rng = np.random.default_rng(SEED)
    sampled = rng.choice(eligible, size=min(N_QUERIES, len(eligible)), replace=False)
    print(f"eligible queries: {len(eligible):,} | sampled: {len(sampled):,}")

    # Trim each sampled query's rows per label to a random target within range
    df_pick = df_esi[df_esi["query_id"].isin(sampled)].copy()
    df_pick["_r"] = rng.random(len(df_pick))
    df_pick["_rank"] = df_pick.groupby(["query_id", "esci_label"])["_r"].rank(method="first").astype(int) - 1
    grp = df_pick.groupby(["query_id", "esci_label"]).size().rename("avail").reset_index()
    grp["lo"] = grp["esci_label"].map(lambda l: LABEL_RULES[l][0])
    grp["hi"] = grp["esci_label"].map(lambda l: LABEL_RULES[l][1])
    grp["hi_eff"] = np.minimum(grp["hi"], grp["avail"])
    grp["target"] = rng.integers(grp["lo"].to_numpy(), (grp["hi_eff"] + 1).to_numpy())
    df_pick = df_pick.merge(grp[["query_id", "esci_label", "target"]], on=["query_id", "esci_label"])
    df_sample = df_pick[df_pick["_rank"] < df_pick["target"]].drop(columns=["_r", "_rank", "target"])
    df_sample = df_sample.reset_index(drop=True)
    print(f"judgement rows: {len(df_sample):,} | products: {df_sample['product_id'].nunique():,}")
    print(df_sample["esci_label"].value_counts().to_dict())

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 2: attach English product text
# Builds the English catalog for the sampled products.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 2: attach English product text (using bundled pt-BR sample).')
else:
    # Attach product text for the sampled products
    cat = df_products[(df_products["product_locale"] == "us") &
                      (df_products["product_id"].isin(df_sample["product_id"].unique()))]
    cat = cat.drop_duplicates("product_id").reset_index(drop=True)
    os.makedirs("out", exist_ok=True)
    print(f"catalog products: {len(cat):,}")

## 3. Translate EN -> pt-BR with Amazon Translate

Amazon Translate batch jobs read documents from S3. JSONL is not a supported format,
so we write the **field values** into `.xlsx` (translated cell-by-cell, structure
preserved), keeping each document under the 1 MB translatable-text cap. Pricing is
per source character ($15 / million).

### 3.1 Brand custom terminology
Generic MT mistranslates brand tokens (e.g. a brand becoming a common word). We build
an Amazon Translate **custom terminology** mapping each catalog brand to itself so
brands survive translation.

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 3: build + import the brand custom terminology
# Creates an Amazon Translate custom terminology so brand names survive
# translation. Calls ImportTerminology (no per-character charge).
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 3: build + import the brand custom terminology (using bundled pt-BR sample).')
else:
    import csv, boto3

    # Build brand terminology CSV (brand -> brand)
    brands = (cat["product_brand"].dropna().map(lambda b: str(b).strip().strip('"').strip()))
    brands = sorted({b for b in brands if b and any(c.isalnum() for c in b)
                     and b.lower() not in {"n/a", "na", "none", "unknown"}
                     and len(b.encode()) <= 200})
    with open("out/brand_terminology.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f); w.writerow(["en", "pt"])
        for b in brands:
            w.writerow([b, b])
    print(f"brand terms: {len(brands):,}")

    translate = boto3.client("translate", region_name=REGION)
    with open("out/brand_terminology.csv", "rb") as f:
        translate.import_terminology(Name="product-brands", MergeStrategy="OVERWRITE",
            TerminologyData={"File": f.read(), "Format": "CSV", "Directionality": "UNI"})
    print("terminology 'product-brands' imported")

### 3.2 Build xlsx documents, upload, and start the batch job

The queries are **deduplicated** before translation (the same query repeats across its
judgements), which avoids tens of thousands of redundant characters.

> This step incurs the Amazon Translate cost. Reduce `N_QUERIES` in step 2 for a cheap
> trial. The full sample is ~US$947.

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 3: build the .xlsx batch documents
# Packs the English text into <1 MB xlsx documents for the batch job.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 3: build the .xlsx batch documents (using bundled pt-BR sample).')
else:
    import xlsxwriter
    from datetime import datetime, timezone

    # Translatable columns only. product_id is deliberately NOT sent to Amazon
    # Translate: it is an opaque ASIN (no text to translate), sending it would be
    # billed per character and could corrupt the id. The ids travel in the
    # manifest instead and are re-attached during reassembly.
    PCOLS = ["product_title", "product_description", "product_bullet_point", "product_color"]
    MAX_DOC_BYTES = 800_000
    XLSX_CT = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"

    def write_xlsx(path, header, rows):
        wb = xlsxwriter.Workbook(path, {"constant_memory": True})
        ws = wb.add_worksheet("data")
        for c, h in enumerate(header):
            ws.write_string(0, c, h)
        for r, row in enumerate(rows, 1):
            for c, v in enumerate(row):
                if v is not None:
                    ws.write_string(r, c, str(v))
        wb.close()

    os.makedirs("out/batch_in", exist_ok=True)
    manifest = {"products": {}, "queries": {}}

    # product chunks
    chunk, ids, b, idx = [], [], 0, 0
    for _, row in cat.iterrows():
        vals = [row.get(c) for c in PCOLS]
        rb = sum(len(str(v).encode()) for v in vals if v is not None)
        if chunk and b + rb > MAX_DOC_BYTES:
            idx += 1; name = f"products_{idx:04d}.xlsx"
            write_xlsx(f"out/batch_in/{name}", PCOLS, chunk); manifest["products"][name] = ids
            chunk, ids, b = [], [], 0
        chunk.append(vals); ids.append(row["product_id"]); b += rb
    if chunk:
        idx += 1; name = f"products_{idx:04d}.xlsx"
        write_xlsx(f"out/batch_in/{name}", PCOLS, chunk); manifest["products"][name] = ids

    # unique queries (single doc assumed; chunk if needed)
    uq = df_sample.drop_duplicates("query")["query"].tolist()
    write_xlsx("out/batch_in/queries_0001.xlsx", ["query"], [[q] for q in uq])
    manifest["queries"]["queries_0001.xlsx"] = uq
    json.dump(manifest, open("out/manifest.json", "w"))
    print(f"documents: {len(manifest['products'])} product + 1 query; unique queries: {len(uq):,}")

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 3: >>> THIS IS THE PAID CELL <<< start the Translate job
# Amazon Translate bills US$15 per million source characters. It also has
# its own PROCEED_WITH_PAID_TRANSLATION = False switch you must flip by
# hand, so it cannot start a paid job by accident on a Run All.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 3: >>> THIS IS THE PAID CELL <<< start the Translate job (using bundled pt-BR sample).')
else:
    # Estimate the Amazon Translate cost for THIS run, then upload and start the job.
    # Amazon Translate bills per source character at $15 / million.
    prod_chars = sum(len(str(v)) for _, row in cat.iterrows() for v in [row.get(c) for c in PCOLS] if v is not None)
    query_chars = sum(len(q) for q in uq)
    total_chars = prod_chars + query_chars
    est_usd = total_chars / 1_000_000 * 15
    print(f"About to translate {total_chars:,} characters "
          f"({len(cat):,} products + {len(uq):,} unique queries).")
    print(f"Estimated Amazon Translate cost for THIS run: ~US${est_usd:,.2f} "
          f"(N_QUERIES={N_QUERIES}). Lower N_QUERIES in Step 2 to reduce it.")

    # The line below starts a paid job. Comment it out to inspect the estimate only.
    PROCEED_WITH_PAID_TRANSLATION = False   # set to True after reviewing the estimate above
    assert PROCEED_WITH_PAID_TRANSLATION, "Set PROCEED_WITH_PAID_TRANSLATION = True to run the paid Translate job."

    s3 = boto3.client("s3", region_name=REGION)
    run_id = f"ptbr-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
    in_prefix = f"translate-in/{run_id}"
    for name in list(manifest["products"]) + list(manifest["queries"]):
        s3.upload_file(f"out/batch_in/{name}", S3_BUCKET, f"{in_prefix}/{name}")

    resp = translate.start_text_translation_job(
        JobName=f"esci-ptbr-{run_id}",
        InputDataConfig={"S3Uri": f"s3://{S3_BUCKET}/{in_prefix}/", "ContentType": XLSX_CT},
        OutputDataConfig={"S3Uri": f"s3://{S3_BUCKET}/translate-out/{run_id}/"},
        DataAccessRoleArn=TRANSLATE_ROLE_ARN,
        SourceLanguageCode="en", TargetLanguageCodes=["pt"],
        TerminologyNames=["product-brands"],
    )
    JOB_ID, OUT_URI = resp["JobId"], f"s3://{S3_BUCKET}/translate-out/{run_id}/"
    print("JobId:", JOB_ID, "| status:", resp["JobStatus"])

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 3: poll the Translate job until it finishes
# Blocks for minutes to a couple of hours while the batch job runs.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 3: poll the Translate job until it finishes (using bundled pt-BR sample).')
else:
    # Poll until the job completes (minutes to a couple of hours)
    while True:
        st = translate.describe_text_translation_job(JobId=JOB_ID)["TextTranslationJobProperties"]["JobStatus"]
        print(st, end=" ", flush=True)
        if st in ("COMPLETED", "COMPLETED_WITH_ERROR", "FAILED", "STOPPED"):
            break
        time.sleep(60)

### 3.3 Reassemble translated output back into JSONL

Amazon Translate preserves the xlsx row order, so we map each translated row back to
its id/query via the manifest, then expand the unique-query translations across every
judgement row.

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 3: download + reassemble the translated xlsx
# Maps translated rows back to product ids / queries via the manifest.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 3: download + reassemble the translated xlsx (using bundled pt-BR sample).')
else:
    from openpyxl import load_workbook
    import urllib.parse

    # find the output subfolder Translate created and download xlsx files
    paginator = s3.get_paginator("list_objects_v2")
    out_key_prefix = OUT_URI.replace(f"s3://{S3_BUCKET}/", "")
    os.makedirs("out/batch_out", exist_ok=True)
    local = {}
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=out_key_prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".xlsx"):
                base = os.path.basename(key)
                orig = base.split(".", 1)[1] if base.startswith("pt.") else base
                dst = f"out/batch_out/{base}"
                s3.download_file(S3_BUCKET, key, dst)
                local[orig] = dst

    def read_rows(path, ncols):
        wb = load_workbook(path, read_only=True); ws = wb.active; out = []
        for i, row in enumerate(ws.iter_rows(values_only=True)):
            if i == 0: continue
            vals = list(row)[:ncols] + [None] * (ncols - len(row)); out.append(vals)
        wb.close(); return out

    prod_pt = {}
    for name, ids in manifest["products"].items():
        for pid, vals in zip(ids, read_rows(local[name], len(PCOLS))):
            prod_pt[pid] = dict(zip(PCOLS, vals)); prod_pt[pid]["product_id"] = pid
    q_pt = {}
    for name, qs in manifest["queries"].items():
        for q, vals in zip(qs, read_rows(local[name], 1)):
            q_pt[q] = vals[0]
    print(f"translated products: {len(prod_pt):,} | unique queries: {len(q_pt):,}")

## 4. Build the labelled dataset and a query-level split

We join translated queries and products back with the ESCI labels, then create a
reproducible **query-level** 70/20/10 train/test/validation split. Splitting by query
(not by row) guarantees no query leaks across partitions.

In [ ]:
# ==========================================================================
# ==== DATA_MODE = "full" ONLY -- STEP 4: join labels and build the query-level split
# Produces `records` from the freshly translated text. In bundled mode
# `records` is loaded from the shipped sample by the loader cell above.
# Skipped automatically when DATA_MODE = "bundled" (the default), which
# loads the pre-translated pt-BR sample instead. Do NOT uncomment or
# force-run this unless you intend to regenerate the pt-BR data yourself.
# ==========================================================================
if DATA_MODE != "full":
    print('DATA_MODE="bundled" -> skipping STEP 4: join labels and build the query-level split (using bundled pt-BR sample).')
else:
    def product_text_pt(p):
        parts = [str(p[c]).strip() for c in ["product_title", "product_bullet_point", "product_description"]
                 if p.get(c) and str(p[c]).strip()]
        return "\n".join(parts) if parts else (p.get("product_color") or "")

    # query-level split
    qids = sorted(df_sample["query_id"].unique())
    rng2 = np.random.default_rng(SEED); rng2.shuffle(qids)
    n = len(qids); n_tr = int(n*0.7); n_te = int(n*0.2)
    split_of = {}
    for i, q in enumerate(qids):
        split_of[q] = "train" if i < n_tr else ("test" if i < n_tr+n_te else "validation")

    records = []
    for _, r in df_sample.iterrows():
        p = prod_pt.get(r["product_id"])
        if not p: continue
        records.append({
            "query_id": int(r["query_id"]),
            "query_pt": q_pt.get(r["query"], r["query"]),
            "product_id": r["product_id"],
            "esci_label": r["esci_label"],
            "split": split_of[r["query_id"]],
            "product_pt": product_text_pt(p),
        })
    import collections
    by_split = collections.Counter(x["split"] for x in records)
    print("rows by split:", dict(by_split))
    print("queries by split:", {s: len({x['query_id'] for x in records if x['split']==s}) for s in ['train','test','validation']})
    json.dump(records, open("out/dataset_ptbr.json", "w"), ensure_ascii=False)

## 4b. Load the pt-BR dataset

This is where the two tracks converge. Both produce the same variable,
`records`: a list of `{query_id, query_pt, product_id, esci_label, split,
product_pt}` dicts.

- `DATA_MODE = "bundled"` reads the pre-translated sample shipped in this
  repository. Nothing above this cell needs to have run.
- `DATA_MODE = "full"` uses the `records` just built in step 4 from your own
  Amazon Translate output.

> **What the bundled sample is.** A 500-query slice of the full derived
> dataset, cut by query so every query keeps its complete E/S/I candidate
> set. It exists so you can run the whole workflow for free. It is **not**
> the dataset behind the published numbers: it retrieves from ~3,072
> products instead of 59,270, which is a much easier task, so expect
> noticeably higher nDCG/Recall than the results table. Reproducing the
> published figures needs the full dataset, which `DATA_MODE = "full"`
> regenerates deterministically from the original English ESCI data.


In [ ]:
import collections

if DATA_MODE == "bundled":
    if not os.path.exists(BUNDLED_SAMPLE):
        raise FileNotFoundError(
            f"bundled sample not found at {BUNDLED_SAMPLE}. Run the notebook "
            "from the folder that contains data/, or set DATA_MODE = 'full'.")
    with open(BUNDLED_SAMPLE, encoding="utf-8") as f:
        records = [json.loads(line) for line in f]
    print(f"loaded bundled pt-BR sample: {len(records):,} rows")
else:
    print(f"using records built in step 4: {len(records):,} rows")

by_split = collections.Counter(r["split"] for r in records)
print("rows by split:", dict(by_split))
print("queries by split:", {s: len({r["query_id"] for r in records
                                    if r["split"] == s})
                            for s in ("train", "test", "validation")})
print("distinct products (retrieval corpus):",
      f'{len({r["product_id"] for r in records}):,}')


## 5. Evaluation harness

For each query we embed the query and product passages, rank products by cosine
similarity (L2-normalized vectors -> dot product), and score against the ESCI labels
with graded gains **E=1.0, S=0.1, I=0.0**.

- **Recall@k** = relevant (E) products retrieved in top-k / total E for the query.
- **nDCG@k** = DCG@k / ideal-DCG@k; order- and grade-aware (our headline metric).
- **Full-corpus** retrieval: each query retrieves from **all** products (realistic
  product search). Unjudged products count as gain 0, which deflates absolute values
  equally for every model, so comparisons stay fair.

In [ ]:
GAIN = {"E": 1.0, "S": 0.1, "I": 0.0}

def dcg(gains):
    g = np.asarray(gains, float)
    return float(np.sum(g / np.log2(np.arange(2, len(g)+2))))

def evaluate_corpus(qids, q_vecs, judg, pids, p_vecs, ks=(5, 10)):
    agg = collections.defaultdict(list); maxk = max(ks)
    for qi, qid in enumerate(qids):
        judged = judg[qid]; label_of = {pid: l for pid, l in judged}
        ideal = sorted((GAIN.get(l, 0.0) for _, l in judged), reverse=True)
        n_rel = sum(1 for _, l in judged if l == "E")
        sims = p_vecs @ q_vecs[qi]
        top = np.argpartition(-sims, min(maxk, len(sims)-1))[:maxk]
        top = top[np.argsort(-sims[top])]
        tp = [pids[i] for i in top]
        for k in ks:
            idcg = dcg(ideal[:k])
            if idcg > 0:
                agg[f"nDCG@{k}"].append(dcg([GAIN.get(label_of.get(p), 0.0) for p in tp[:k]]) / idcg)
            if n_rel:
                agg[f"Recall@{k}"].append(sum(1 for p in tp[:k] if label_of.get(p) == "E") / n_rel)
    return {m: float(np.mean(v)) for m, v in agg.items()}

# Build test-split structures + full product corpus
test_q = {r["query_id"]: r["query_pt"] for r in records if r["split"] == "test"}
judg = collections.defaultdict(list)
for r in records:
    if r["split"] == "test":
        judg[r["query_id"]].append((r["product_id"], r["esci_label"]))
corpus = {}
for r in records:
    corpus.setdefault(r["product_id"], r["product_pt"])
test_qids = sorted(test_q); corpus_pids = sorted(corpus)
print(f"test queries: {len(test_qids):,} | corpus products: {len(corpus_pids):,}")

In [ ]:
# Embedders: e5 (local sentence-transformers) and Titan (Bedrock)
from sentence_transformers import SentenceTransformer

def e5_embed(model_id_or_path, texts, prefix, max_seq_length=256, bs=64):
    model = SentenceTransformer(model_id_or_path)
    model.max_seq_length = max_seq_length
    return model.encode([f"{prefix}{t}" for t in texts], batch_size=bs,
                        normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

def titan_embed(texts, dims=1024, workers=8):
    from concurrent.futures import ThreadPoolExecutor
    rt = boto3.client("bedrock-runtime", region_name=REGION)
    def one(t):
        last = None
        for a in range(8):
            try:
                body = json.dumps({"inputText": t[:48000] or " ", "dimensions": dims, "normalize": True})
                r = rt.invoke_model(modelId="amazon.titan-embed-text-v2:0", body=body)
                return np.asarray(json.loads(r["body"].read())["embedding"], np.float32)
            except Exception as e:
                last = e                      # keep the cause: throttling and an
                time.sleep(min(2**a, 20))     # access/credentials error look alike
        raise RuntimeError(f"Titan invoke_model failed after 8 attempts: "
                           f"{type(last).__name__}: {last}")
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return np.vstack(list(ex.map(one, texts)))

## 6. Establish reference points: base e5-small and Titan

We embed the test queries and the full product catalog with the base e5-small
model (locally) and with Amazon Titan Text Embeddings V2 (on Amazon Bedrock), and
evaluate both on full-corpus retrieval. These give us reference points to
contextualize the fine-tuning gains measured in step 8.

In [ ]:
# Base e5-small
qv = e5_embed("intfloat/multilingual-e5-small", [test_q[q] for q in test_qids], "query: ")
pv = e5_embed("intfloat/multilingual-e5-small", [corpus[p] for p in corpus_pids], "passage: ")
base_res = evaluate_corpus(test_qids, qv, judg, corpus_pids, pv)
print("base e5-small:", base_res)

In [ ]:
# Titan V2 (Bedrock) - makes ~ (#queries + #products) API calls.
# Optional: needs model access to amazon.titan-embed-text-v2:0 in Amazon Bedrock.
# On failure this cell reports the cause and continues, so "Run All" survives an
# account without Bedrock access; the comparison table then omits the Titan row.
if not RUN_TITAN:
    print("RUN_TITAN = False -> skipping the Titan reference point (step 6).")
else:
    try:
        qvt = titan_embed([test_q[q] for q in test_qids])
        pvt = titan_embed([corpus[p] for p in corpus_pids])
        titan_res = evaluate_corpus(test_qids, qvt, judg, corpus_pids, pvt)
        print("titan v2:", titan_res)
    except Exception as e:
        print(f"[skipped] Titan reference point unavailable -> {type(e).__name__}: {e}")
        print(f"Request access to amazon.titan-embed-text-v2:0 in Amazon Bedrock for "
              f"REGION={REGION}, or set RUN_TITAN = False to skip this step on purpose.")


## 7. Fine-tune e5-small on Amazon SageMaker

We build `(query, positive E product, hard-negative I product)` triplets from the
**train** split and fine-tune with `MultipleNegativesRankingLoss` (in-batch negatives
+ the explicit hard negative). Substitutes (S) are excluded from training. Training
runs as an **Amazon SageMaker training job** on a single GPU (`ml.g5.xlarge`); the job
auto-terminates when done, so there is no idle cost. A validation
`InformationRetrievalEvaluator` selects the highest-scoring checkpoint on validation.

In [ ]:
# 7.1 Prepare triplets + validation IR set locally, with e5 prefixes baked in.
# finetune_src/ must exist BEFORE the %%writefile cells below: the
# writefile magic does not create parent directories.
os.makedirs("finetune_src", exist_ok=True)
os.makedirs("finetune_data/train", exist_ok=True)
os.makedirs("finetune_data/validation", exist_ok=True)

train_by_q = collections.defaultdict(list); qtext = {}; ptext = {}
for r in records:
    if r["split"] != "train": continue
    train_by_q[r["query_id"]].append((r["product_id"], r["esci_label"]))
    qtext[r["query_id"]] = r["query_pt"]; ptext[r["product_id"]] = r["product_pt"]

rngt = random.Random(SEED)
with open("finetune_data/train/train_triplets.jsonl", "w", encoding="utf-8") as f:
    n_tri = 0
    for qid, judged in train_by_q.items():
        pos = [p for p, l in judged if l == "E"]; neg = [p for p, l in judged if l == "I"]
        if not pos or not neg: continue
        for p in pos:
            f.write(json.dumps({"anchor": f"query: {qtext[qid]}",
                                "positive": f"passage: {ptext[p]}",
                                "negative": f"passage: {ptext[rngt.choice(neg)]}"}, ensure_ascii=False) + "\n")
            n_tri += 1
print(f"training triplets: {n_tri:,}")

val_q, val_corpus, val_rel = {}, {}, {}
for r in records:
    if r["split"] != "validation": continue
    qid = str(r["query_id"])
    if r["esci_label"] == "E":
        val_rel.setdefault(qid, []).append(str(r["product_id"]))
        val_q[qid] = f"query: {r['query_pt']}"
    val_corpus[str(r["product_id"])] = f"passage: {r['product_pt']}"
val_q = {q: val_q[q] for q in val_rel}  # keep only queries with a positive
json.dump({"queries": val_q, "corpus": val_corpus, "relevant_docs": val_rel},
          open("finetune_data/validation/val_ir.json", "w"), ensure_ascii=False)
print(f"validation: {len(val_q):,} queries, {len(val_corpus):,} corpus docs")

In [ ]:
%%writefile finetune_src/train.py
import argparse, json, os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from sentence_transformers import InputExample, SentenceTransformer, losses
from transformers import set_seed
set_seed(42)  # reproducibility (up to minor GPU nondeterminism)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader

def load_triplets(ch):
    ex = []
    with open(os.path.join(ch, "train_triplets.jsonl"), encoding="utf-8") as f:
        for line in f:
            o = json.loads(line)
            ex.append(InputExample(texts=[o["anchor"], o["positive"], o["negative"]]))
    return ex

def build_eval(ch):
    p = os.path.join(ch, "val_ir.json") if ch else None
    if not p or not os.path.exists(p):
        return None
    d = json.load(open(p, encoding="utf-8"))
    rel = {q: set(v) for q, v in d["relevant_docs"].items()}
    return InformationRetrievalEvaluator(queries=d["queries"], corpus=d["corpus"],
        relevant_docs=rel, name="ptbr-val", show_progress_bar=False,
        precision_recall_at_k=[5, 10], ndcg_at_k=[5, 10], mrr_at_k=[10])

ap = argparse.ArgumentParser()
ap.add_argument("--model-id", default="intfloat/multilingual-e5-small")
ap.add_argument("--epochs", type=int, default=2)
ap.add_argument("--batch-size", type=int, default=32)
ap.add_argument("--max-seq-length", type=int, default=256)
ap.add_argument("--lr", type=float, default=2e-5)
ap.add_argument("--warmup-ratio", type=float, default=0.1)
ap.add_argument("--train", default=os.environ.get("SM_CHANNEL_TRAIN"))
ap.add_argument("--validation", default=os.environ.get("SM_CHANNEL_VALIDATION"))
ap.add_argument("--model-dir", default=os.environ.get("SM_MODEL_DIR", "model"))
a = ap.parse_args()

model = SentenceTransformer(a.model_id)
model.max_seq_length = a.max_seq_length
ex = load_triplets(a.train)
print("triplets:", len(ex))
loader = DataLoader(ex, shuffle=True, batch_size=a.batch_size)
loss = losses.MultipleNegativesRankingLoss(model)
ev = build_eval(a.validation)
steps = max(1, len(loader))
model.fit(train_objectives=[(loader, loss)], evaluator=ev, epochs=a.epochs,
          warmup_steps=int(steps * a.epochs * a.warmup_ratio),
          optimizer_params={"lr": a.lr}, evaluation_steps=steps,
          save_best_model=bool(ev), output_path=a.model_dir, use_amp=True)
if not ev:
    model.save(a.model_dir)
print("saved ->", a.model_dir)

In [ ]:
%%writefile finetune_src/requirements.txt
sentence-transformers==2.7.0

In [ ]:
# 7.2 Launch the SageMaker training job (auto-terminates when done).
# This step is PAID: ~US$0.27 for ~11 minutes on one ml.g5.xlarge. It needs
# S3_BUCKET and a SageMaker execution role (auto-detected inside SageMaker
# Studio). Set RUN_TRAINING = False in the first cell to skip it.
if not RUN_TRAINING:
    print("RUN_TRAINING = False -> skipping the SageMaker fine-tuning job (step 7.2).")
elif S3_BUCKET == "REPLACE_ME":
    raise ValueError(
        "S3_BUCKET is still 'REPLACE_ME'. Fine-tuning uploads the training data to "
        "Amazon S3, so set S3_BUCKET (and SAGEMAKER_ROLE_ARN, unless you are running "
        "inside Amazon SageMaker Studio) in the first cell -- or set "
        "RUN_TRAINING = False to skip step 7.2.")
else:
    import sagemaker, tarfile, boto3
    from sagemaker.huggingface import HuggingFace

    sess = sagemaker.Session()
    try:
        role = sagemaker.get_execution_role()          # works inside SageMaker Studio
    except Exception:
        role = SAGEMAKER_ROLE_ARN                        # else use the configured role

    train_s3 = sess.upload_data("finetune_data/train", bucket=S3_BUCKET, key_prefix="esci-ft/train")
    val_s3 = sess.upload_data("finetune_data/validation", bucket=S3_BUCKET, key_prefix="esci-ft/validation")

    estimator = HuggingFace(
        entry_point="train.py", source_dir="finetune_src", role=role,
        instance_type="ml.g5.xlarge", instance_count=1,
        transformers_version="4.36", pytorch_version="2.1", py_version="py310",
        hyperparameters={"model-id": "intfloat/multilingual-e5-small", "epochs": 2,
                         "batch-size": 32, "max-seq-length": 256, "lr": 2e-5, "warmup-ratio": 0.1},
    )
    estimator.fit({"train": train_s3, "validation": val_s3})   # blocks until COMPLETED

    # Download and extract the trained model artifact
    b, k = estimator.model_data.replace("s3://", "").split("/", 1)
    os.makedirs("out/e5-small-ptbr-ft", exist_ok=True)
    boto3.client("s3", region_name=REGION).download_file(b, k, "out/model.tar.gz")
    # filter="data" blocks path-traversal entries (CVE-2007-4559) and is the
    # default from Python 3.14; passing it explicitly keeps 3.9-3.13 quiet too.
    with tarfile.open("out/model.tar.gz") as tf:
        try:
            tf.extractall("out/e5-small-ptbr-ft", filter="data")
        except TypeError:  # Python < 3.12 has no filter kwarg
            tf.extractall("out/e5-small-ptbr-ft")
    print("fine-tuned model -> out/e5-small-ptbr-ft")


## 8. Evaluate the fine-tuned model and compare

In [ ]:
# Compare whatever is available. The fine-tuned row needs step 7.2 (Amazon
# SageMaker, RUN_TRAINING) and the Titan row needs step 6 (Amazon Bedrock,
# RUN_TITAN); if you skipped either, the table omits that row instead of failing.
import pandas as pd

FT_MODEL = "out/e5-small-ptbr-ft"
results = {"e5-small (base)": (384, base_res)}

if os.path.isdir(FT_MODEL):
    qvf = e5_embed(FT_MODEL, [test_q[q] for q in test_qids], "query: ")
    pvf = e5_embed(FT_MODEL, [corpus[p] for p in corpus_pids], "passage: ")
    ft_res = evaluate_corpus(test_qids, qvf, judg, corpus_pids, pvf)
    results["e5-small (fine-tuned)"] = (384, ft_res)
else:
    print(f"[skipped] fine-tuned model not found at {FT_MODEL} -- run step 7.2 "
          f"(RUN_TRAINING = True)")

if "titan_res" in globals():
    results["Titan Text Embeddings V2"] = (1024, titan_res)
else:
    print("[skipped] Titan reference point -- step 6 (Amazon Bedrock) not run")

table = pd.DataFrame([{"model": n, "dims": d,
                       **{k: round(r[k], 4) for k in sorted(r)}}
                      for n, (d, r) in results.items()])
print(table.to_string(index=False))

if DATA_MODE == "bundled":
    print(f"\nNote: DATA_MODE='bundled' retrieves from "
          f"{len(corpus_pids):,} products, not the 59,270 of the full "
          f"dataset. A smaller corpus is an easier task, so these numbers "
          f"run HIGHER than the published results table.")


## Results

Full-corpus retrieval on the held-out test set (2,000 queries, 59,270 products). Your
exact numbers may vary slightly by seed and hardware:

| Model | dims | nDCG@10 | Recall@10 |
| --- | --- | --- | --- |
| e5-small (base) | 384 | ~0.276 | ~0.345 |
| **e5-small (fine-tuned)** | **384** | **~0.432** | **~0.528** |
| Titan Text Embeddings V2 | 1024 | ~0.373 | ~0.458 |

Observations specific to this task (pt-BR product retrieval) and evaluation:

- **Task- and language-specific fine-tuning increased the task metrics** by ~56% relative
  nDCG@10, using labeled domain data with the same architecture and dimensionality.
- **The general-purpose reference point produced the results shown without training.**
  Amazon Titan Text Embeddings V2 provides multilingual representations out of the box and
  is a fit when you need coverage without training. On this product-retrieval task in
  pt-BR, the fine-tuned compact model scored higher on the task metrics and produced
  vectors ~2.7x smaller (384 vs 1,024 dimensions), which reduces index and serving cost.

The takeaway is not that one model is better in general; specializing a compact model for
a specific task and language changed results on that task by the margins above.

## Clean up

Delete resources you created to avoid ongoing charges: any SageMaker inference endpoint
used to serve the model (training jobs stop billing automatically when they complete),
and the S3 objects created for the translation job if no longer needed.

## Conclusion

Amazon Titan Text Embeddings is a general-purpose, multilingual option that runs without
training. When you have a focused task such as product retrieval in a specific language
and labeled domain data, fine-tuning a compact open model on Amazon SageMaker can
specialize it for that task and language while keeping the serving footprint small. We
demonstrated this for pt-BR; the same workflow reproduces for any language by changing
the Amazon Translate target.

## Attribution

Data: Shopping Queries Dataset (ESCI), Amazon Science, `amazon-research/esci-code`,
Apache-2.0, arXiv:2206.06588. The pt-BR text is machine-translated with Amazon Translate
and may contain errors. Cite the original authors and retain the Apache-2.0 license (`data/LICENSE`) and `NOTICE`
when redistributing the data. Absolute retrieval numbers are deflated because unjudged products count
as irrelevant (consistent across models); we use a custom query-level split; single seed,
single run.